Konteks / Skenario
Manajemen "Kopi Nusantara" (skenario yang sama dari Pertemuan 7, 9, dan 10) ingin memiliki alat bantu prediksi omzet harian tiap cabang, agar dapat merencanakan stok bahan baku dengan lebih baik. Anda ditugaskan membangun model regresi untuk memprediksi total_penjualan sebuah pesanan berdasarkan atribut-atributnya.

Menyiapkan Dataset
Jalankan cell berikut untuk membuat dataset.

In [1]:
import numpy as np
import pandas as pd

np.random.seed(111)
n = 2500
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas11 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "harga_satuan": np.random.choice([15000, 18000, 20000, 22000, 25000, 30000], size=n),
})
data_tugas11["total_penjualan"] = data_tugas11["qty"] * data_tugas11["harga_satuan"] + np.random.normal(0, 8000, size=n)
data_tugas11.to_csv("tugas11_penjualan.csv", index=False)

print(f"Dataset berhasil dibuat: {len(data_tugas11)} baris")
print(data_tugas11.head())

Dataset berhasil dibuat: 2500 baris
        nama_menu      cabang waktu_pesan  qty  harga_satuan  total_penjualan
0  Kentang Goreng    Semarang       Malam    1         30000     27750.163532
1  Kentang Goreng  Yogyakarta        Sore    3         20000     64216.160744
2  Kentang Goreng    Semarang       Siang    5         22000    108722.792592
3  Kentang Goreng  Yogyakarta        Pagi    2         20000     54483.205872
4      Roti Bakar    Magelang       Malam    2         30000     58157.944120


Inisialisasi SparkSession & Membaca Dataset

In [2]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import col

# Inisialisasi SparkSession
spark = SparkSession.builder.appName("Tugas11_MLlib").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

# Membaca file CSV
df = spark.read.csv("tugas11_penjualan.csv", header=True, inferSchema=True)
print("Total baris data Spark:", df.count())
df.show(5)

26/11/12 17:13:39 WARN Utils: Your hostname, adnan resolves to a loopback address: 127.0.1.1; using 10.90.69.75 instead (on interface wlp2s0)
26/11/12 17:13:39 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/12 17:13:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Total baris data Spark: 2500
+--------------+----------+-----------+---+------------+------------------+
|     nama_menu|    cabang|waktu_pesan|qty|harga_satuan|   total_penjualan|
+--------------+----------+-----------+---+------------+------------------+
|Kentang Goreng|  Semarang|      Malam|  1|       30000|27750.163531558544|
|Kentang Goreng|Yogyakarta|       Sore|  3|       20000| 64216.16074371864|
|Kentang Goreng|  Semarang|      Siang|  5|       22000|108722.79259154423|
|Kentang Goreng|Yogyakarta|       Pagi|  2|       20000|54483.205871792496|
|    Roti Bakar|  Magelang|      Malam|  2|       30000| 58157.94412015849|
+--------------+----------+-----------+---+------------+------------------+
only showing top 5 rows



A. Feature Engineering (bobot 20%)

Gunakan StringIndexer untuk mengubah kolom nama_menu, cabang, dan waktu_pesan menjadi indeks numerik. Gabungkan seluruh fitur (qty, harga_satuan, dan ketiga kolom hasil indexing) menjadi satu kolom features menggunakan VectorAssembler.

In [3]:
# 1. StringIndexer untuk ketiga kolom kategorikal
indexer_menu = StringIndexer(inputCol="nama_menu", outputCol="nama_menu_idx", handleInvalid="skip")
indexer_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx", handleInvalid="skip")
indexer_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_pesan_idx", handleInvalid="skip")

# Fit dan transform secara berurutan (simpan model indexer untuk dipakai di Bagian E)
model_menu = indexer_menu.fit(df)
df_indexed = model_menu.transform(df)

model_cabang = indexer_cabang.fit(df_indexed)
df_indexed = model_cabang.transform(df_indexed)

model_waktu = indexer_waktu.fit(df_indexed)
df_indexed = model_waktu.transform(df_indexed)

# 2. VectorAssembler untuk menggabungkan 5 fitur
assembler_utama = VectorAssembler(
    inputCols=["qty", "harga_satuan", "nama_menu_idx", "cabang_idx", "waktu_pesan_idx"],
    outputCol="features"
)

df_features = assembler_utama.transform(df_indexed)
df_features.select("qty", "harga_satuan", "nama_menu_idx", "cabang_idx", "waktu_pesan_idx", "features").show(5, truncate=False)

+---+------------+-------------+----------+---------------+-------------------------+
|qty|harga_satuan|nama_menu_idx|cabang_idx|waktu_pesan_idx|features                 |
+---+------------+-------------+----------+---------------+-------------------------+
|1  |30000       |5.0          |2.0       |1.0            |[1.0,30000.0,5.0,2.0,1.0]|
|3  |20000       |5.0          |1.0       |3.0            |[3.0,20000.0,5.0,1.0,3.0]|
|5  |22000       |5.0          |2.0       |2.0            |[5.0,22000.0,5.0,2.0,2.0]|
|2  |20000       |5.0          |1.0       |0.0            |[2.0,20000.0,5.0,1.0,0.0]|
|2  |30000       |4.0          |0.0       |1.0            |[2.0,30000.0,4.0,0.0,1.0]|
+---+------------+-------------+----------+---------------+-------------------------+
only showing top 5 rows



B. Train/Test Split & Pelatihan Model (bobot 20%)

Bagi data menjadi 75% training dan 25% testing (seed=101). Latih model LinearRegression dengan labelCol="total_penjualan".

In [4]:
# Train/Test Split (75% training, 25% testing dengan seed=101)
train_data, test_data = df_features.randomSplit([0.75, 0.25], seed=101)

# Inisialisasi dan latih model LinearRegression
lr = LinearRegression(featuresCol="features", labelCol="total_penjualan")
lr_model = lr.fit(train_data)

print("Model LinearRegression berhasil dilatih!")
print(f"Koefisien: {lr_model.coefficients}")
print(f"Intercept: {lr_model.intercept}")

Model LinearRegression berhasil dilatih!
Koefisien: [22046.663271025758,3.022920868160672,-123.21425598657399,279.28452461677676,339.6387953231708]
Intercept: -66903.17766485889


C. Evaluasi Model (bobot 20%)

Hitung dan tampilkan RMSE dan R² dari model pada data testing. Tulis 1-2 kalimat interpretasi: apakah model ini cukup baik untuk dipakai memprediksi omzet?

In [5]:
# Melakukan prediksi pada data testing
predictions_utama = lr_model.transform(test_data)

# Evaluasi menggunakan RegressionEvaluator
evaluator_rmse = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="rmse")
evaluator_r2 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2")

rmse = evaluator_rmse.evaluate(predictions_utama)
r2 = evaluator_r2.evaluate(predictions_utama)

print(f"Evaluasi Model Utama (5 Fitur):")
print(f"- RMSE : {rmse}")
print(f"- R2   : {r2}")

Evaluasi Model Utama (5 Fitur):
- RMSE : 10900.803581280325
- R2   : 0.910679109181288


D. Eksperimen Perbandingan Fitur (bobot 25%)

Latih model kedua yang HANYA menggunakan fitur qty dan harga_satuan (tanpa informasi menu/cabang/waktu). Bandingkan R² model kedua ini dengan model utama pada bagian C. Tulis kesimpulan: apakah menambahkan informasi menu/cabang/waktu benar-benar meningkatkan akurasi prediksi secara berarti?


In [6]:
# 1. VectorAssembler khusus 2 fitur (tanpa indeks kategori)
assembler_kedua = VectorAssembler(
    inputCols=["qty", "harga_satuan"],
    outputCol="features_kedua"
)
df_features_kedua = assembler_kedua.transform(df)

# 2. Split data dengan train/test split yang sama (seed=101)
train_k2, test_k2 = df_features_kedua.randomSplit([0.75, 0.25], seed=101)

# 3. Latih model LinearRegression kedua
lr_kedua = LinearRegression(featuresCol="features_kedua", labelCol="total_penjualan")
model_kedua = lr_kedua.fit(train_k2)

# 4. Evaluasi model kedua
pred_kedua = model_kedua.transform(test_k2)
r2_kedua = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2").evaluate(pred_kedua)

print(f"R2 Model Utama (5 Fitur)  : {r2}")
print(f"R2 Model Kedua (2 Fitur)  : {r2_kedua}")

R2 Model Utama (5 Fitur)  : 0.910679109181288
R2 Model Kedua (2 Fitur)  : 0.9107572632054388


E. Prediksi Skenario Baru (bobot 15%)

Buat sebuah DataFrame baru berisi 3 skenario pesanan (data dummy) (boleh bebas kombinasi menu/cabang/waktu/qty/harga, namun harus melalui tahapan StringIndexer dan VectorAssembler yang sama seperti data training — perhatikan: gunakan model StringIndexer yang sama yang sudah di-fit() sebelumnya, jangan membuat StringIndexer baru). Tampilkan hasil prediksi total_penjualan untuk ketiga skenario tersebut.


In [7]:
# 1. Membuat DataFrame data dummy untuk 3 skenario pesanan baru menggunakan Pandas, lalu ubah ke Spark DataFrame
scenarios_pd = pd.DataFrame({
    "nama_menu": ["Kopi Susu", "Americano", "Roti Bakar"],
    "cabang": ["Magelang", "Yogyakarta", "Semarang"],
    "waktu_pesan": ["Pagi", "Siang", "Malam"],
    "qty": [2, 4, 3],
    "harga_satuan": [20000, 18000, 25000]
})
df_scenarios = spark.createDataFrame(scenarios_pd)

# 2. Transformasi menggunakan StringIndexer yang SAMA (model_menu, model_cabang, model_waktu)
df_scenarios_indexed = model_menu.transform(df_scenarios)
df_scenarios_indexed = model_cabang.transform(df_scenarios_indexed)
df_scenarios_indexed = model_waktu.transform(df_scenarios_indexed)

# 3. Transformasi menggunakan VectorAssembler utama
df_scenarios_features = assembler_utama.transform(df_scenarios_indexed)

# 4. Melakukan prediksi menggunakan lr_model utama
final_predictions = lr_model.transform(df_scenarios_features)

# 5. Tampilkan hasil prediksi skenario baru
final_predictions.select(
    "nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan", "prediction"
).show(truncate=False)

+----------+----------+-----------+---+------------+-----------------+
|nama_menu |cabang    |waktu_pesan|qty|harga_satuan|prediction       |
+----------+----------+-----------+---+------------+-----------------+
|Kopi Susu |Magelang  |Pagi       |2  |20000       |37525.35198441948|
|Americano |Yogyakarta|Siang      |4  |18000       |76654.61316139936|
|Roti Bakar|Semarang  |Malam      |3  |25000       |75215.1846728456 |
+----------+----------+-----------+---+------------+-----------------+

